In [ ]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load PDF

In [2]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Sample Content:\n{pages[0].page_content[:500]}")

Total Pages: 30
Sample Content:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director Ge


## Step 2: Split into Chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Chunks: {len(chunks)}")
print(f"Sample Chunk:\n{chunks[0].page_content}")

Total Chunks: 136
Sample Chunk:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director General of Defence Research and
Development Organisation
In office
1992–1999
A. P. J. Abdul Kalam
Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl
kəˈlɑːm/ ⓘ UB-duul k ə -LAHM; 15 October 1931 –
27 July 2015) was an Indian aerospace scientist and
statesman who served as the president of India from
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a


## Step 3: Create Embeddings & Store in FAISS

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


## Step 4: Create Retriever

In [5]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

## Step 5: Setup LLM

In [6]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

## Step 6: Generate Hypothetical Answer (HyDE)

Instead of embedding the raw query, we first ask the LLM to generate a **hypothetical answer**. This fabricated-but-plausible answer is closer in embedding space to the real documents than a short question would be.

In [7]:
query = "What was Abdul Kalam's role in India's missile programme?"

# Step 6a: Generate hypothetical answer
hyde_prompt = f"""Answer the following question in 3-4 sentences. 
You can make up a plausible answer even if you're not sure.

Question: {query}
Answer:"""

hypothetical_answer = llm.invoke(hyde_prompt)
print("Hypothetical Answer:\n", hypothetical_answer.content)

Hypothetical Answer:
 Abdul Kalam played a crucial role in India's missile programme as the Chief Executive of the Defence Research and Development Organisation (DRDO) from 1992 to 2007. During his tenure, he oversaw the development of several indigenous missile systems, including the Agni and BrahMos missiles. He was also instrumental in the development of the Prithvi and Agni-III missiles, which were designed to counter the threat of nuclear-armed Pakistan and China. Under his leadership, India successfully tested and deployed several missiles, marking a significant milestone in the country's military capabilities.


## Step 7: Embed Hypothetical Answer & Retrieve Real Chunks

In [8]:
# Embed the hypothetical answer instead of the raw query
hyde_embedding = embeddings.embed_query(hypothetical_answer.content)

# Retrieve real chunks using the hypothetical answer's embedding
retrieved_docs = vector_store.similarity_search_by_vector(hyde_embedding, k=3)

print(f"Retrieved {len(retrieved_docs)} chunks using HyDE")

Retrieved 3 chunks using HyDE


## Step 8: Generate Final Grounded Answer

In [9]:
# Build context from retrieved chunks
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# Create final prompt with real context
final_prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

# Get grounded response
response = llm.invoke(final_prompt)
print("Final Answer:", response.content)

Final Answer: Abdul Kalam played a major role in the development of India's missiles, including Agni and Prithvi, and was known as the "Missile Man of India" for his work on ballistic missile and launch vehicle technology.


## Retrieved Source Chunks

In [11]:
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Chunk {i+1} (Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content)


--- Chunk 1 (Page 1) ---
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a
scientist and science administrator, mainly at the
Defence Research and Development Organisation
(DRDO) and Indian Space Research Organisation
(ISRO) and was intimately involved in India's civilian
space programme and military missile development
efforts. He was known as the "Missile Man of India"
for his work on the development of ballistic missile
and launch vehicle technology. He also played a
pivotal organisational, technical, and political role in
Pokhran-II nuclear tests in 1998, India's second such
test after the first test in 1974.
Kalam was elected as the president of India in 2002
with the support of both the ruling Bharatiya Janata
Party and the then-opposition Indian National
Congress. He was widely referred to as the "People's
President". He engaged in teaching, writing and public

--- Chu

## Test Questions

1. What year was Abdul Kalam born?
2. What was the name of the coronary stent Kalam co-developed?
3. Which institution did Kalam study aerospace engineering at?
4. How many electoral votes did Kalam receive in the 2002 presidential election?
5. What was Kalam's role at DRDO from 1992 to 1999?

Hypothetical Answer:
 Abdul Kalam played a crucial role in India's missile programme as the Chief Executive of the Defence Research and Development Organisation (DRDO) from 1992 to 2007. During his tenure, he oversaw the development of several indigenous missile systems, including the Agni and BrahMos missiles. He was also instrumental in the development of the Prithvi and Agni-III missiles, which were designed to counter the threat of nuclear-armed Pakistan and China. Under his leadership, India successfully tested and deployed several missiles, marking a significant milestone in the country's military capabilities.

Final Answer: Abdul Kalam played a major role in the development of India's missiles, including Agni and Prithvi, and was known as the "Missile Man of India" for his work on ballistic missile and launch vehicle technology.